# 基于医学规则排牙1.0

调用基于医学规则排牙工作流后，如果医生需要调整工单，调整后的工单可以再使用工单排牙重排。

In [1]:
# 导入必要的包以及定义函数
import os
import time
import requests
import json
import trimesh
import urllib
import numpy as np

# 定义调用规则

请根据您从我方获取的信息修改以下代码块

In [2]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


In [3]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

## 基于医学规则排牙1.0
该工作流通过自动分析输入数据生成加工单并根据加工单进行自动排牙，相当于自动工单与工单排牙的整合。https://www.chohotech.com/docs/cloud-zh/#/workflow/oral-arrangement-medical-1

下面是pre_form，您可以通过auto-form自动工单获得，也可以类似下面自定义输入

In [4]:
pre_form_config = {
    "remove_teeth_set": [],  # 不指定拔牙，让系统自动判断
    "gap": None,  # 自动预留间隙
    "locked_teeth_set": [],  # 不锁定牙齿
    "x_axis_position": ["U", "L"],  # 允许上下颌扩弓
    "IPR": {
        "U": ["front", "left_back", "right_back"],  # 上颌允许IPR
        "L": ["front", "left_back", "right_back"]   # 下颌允许IPR
    },
    "molar_movement": {
        "U": ["left", "right"],  # 上颌允许双侧后牙移动
        "L": ["left", "right"]   # 下颌允许双侧后牙移动
    }
}

In [ ]:
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-arrangement-medical",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "upper_mesh": {"type":"drc", "data": upload_file("upper_jaw_scan.drc")},
      "lower_mesh": {"type":"ply", "data": upload_file("lower_jaw_scan.ply")},
      "ceph": upload_file("ceph.jpg"),
      "smile_photo": upload_file("face_smile.jpg"),
      "pre_form" : json.dumps(pre_form_config)  # 可选填的预工单
  },
  'output_config': {
      "teeth_comp": {"type": "ply"},        # 邻接面补全后牙齿
      "arranged_comp": {"type": "ply"},     # 排牙结果
      "upper_mesh": {"type": "ply"},        # 预处理上颌
      "lower_mesh": {"type": "ply"},        # 预处理下颌
      "align_matrix": {},                   # 对齐矩阵
      "transformation_dict": {},            # 变换字典
      "form": {}                            # 生成的工单
  }
}
result_arrangement = run_job_and_get_results(json_call, 1200)

workflow id is wf_1766576607-5ca47413-a950-405e-ad07-048a7f004f70
API finished in 197.15209674835205s


In [7]:
print(f"输出包含的键: {list(result_arrangement.keys())}")

输出包含的键: ['align_matrix', 'arranged_comp', 'form', 'l_align_matrix', 'l_axis', 'l_teeth_comp', 'lower_mesh', 'lower_seg_label', 'transformation_dict', 'u_align_matrix', 'u_axis', 'u_teeth_comp', 'upper_mesh', 'upper_seg_label']


In [ ]:
# 排牙结果
sum([retrieve_mesh(result_arrangement['arranged_comp'][k]) for k in result_arrangement['arranged_comp'].keys()], None).show()